In [ ]:
# ==============================================================================
# BLOCK 1: DEPENDENCIES & INGESTION ENGINE
# ==============================================================================
!pip install -q requests pandas pyarrow fastparquet tqdm openmeteo-requests requests_cache retry_requests

import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm

# Configure retry session to avoid API timeouts
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.5)

print("Dependencies configured. Data engine initialized.")

Dependencies configured. Data engine initialized.


In [ ]:
# ==============================================================================
# BLOCK 1: COMPLETE PAN-INDIA DISTRICT REPOSITORY
# ==============================================================================
import pandas as pd
import numpy as np

# Exhaustive multi-district distribution across all 28 States & UTs of India
PAN_INDIA_DISTRICTS = [
    # --- NORTH ZONE ---
    {"district": "Ludhiana", "state": "Punjab", "latitude": 30.9010, "longitude": 75.8573, "zone": "Trans-Gangetic"},
    {"district": "Amritsar", "state": "Punjab", "latitude": 31.6340, "longitude": 74.8723, "zone": "Trans-Gangetic"},
    {"district": "Bathinda", "state": "Punjab", "latitude": 30.2110, "longitude": 74.9455, "zone": "Trans-Gangetic"},
    {"district": "Karnal", "state": "Haryana", "latitude": 29.6857, "longitude": 76.9905, "zone": "Trans-Gangetic"},
    {"district": "Hisar", "state": "Haryana", "latitude": 29.1492, "longitude": 75.7217, "zone": "Trans-Gangetic"},
    {"district": "Sirsa", "state": "Haryana", "latitude": 29.5349, "longitude": 75.0298, "zone": "Trans-Gangetic"},
    {"district": "Meerut", "state": "Uttar Pradesh", "latitude": 28.9845, "longitude": 77.7064, "zone": "Upper Gangetic"},
    {"district": "Agra", "state": "Uttar Pradesh", "latitude": 27.1767, "longitude": 78.0081, "zone": "Upper Gangetic"},
    {"district": "Bareilly", "state": "Uttar Pradesh", "latitude": 28.3670, "longitude": 79.4304, "zone": "Upper Gangetic"},
    {"district": "Varanasi", "state": "Uttar Pradesh", "latitude": 25.3176, "longitude": 82.9739, "zone": "Middle Gangetic"},
    {"district": "Gorakhpur", "state": "Uttar Pradesh", "latitude": 26.7606, "longitude": 83.3732, "zone": "Middle Gangetic"},
    {"district": "Jhansi", "state": "Uttar Pradesh", "latitude": 25.4484, "longitude": 78.5685, "zone": "Central Plateau"},
    {"district": "Prayagraj", "state": "Uttar Pradesh", "latitude": 25.4358, "longitude": 81.8463, "zone": "Middle Gangetic"},

    # --- HIMALAYAN & MOUNTAIN REGIONS ---
    {"district": "Srinagar", "state": "Jammu and Kashmir", "latitude": 34.0837, "longitude": 74.7973, "zone": "Western Himalayan"},
    {"district": "Anantnag", "state": "Jammu and Kashmir", "latitude": 33.7311, "longitude": 75.1522, "zone": "Western Himalayan"},
    {"district": "Jammu", "state": "Jammu and Kashmir", "latitude": 32.7266, "longitude": 74.8570, "zone": "Western Himalayan"},
    {"district": "Shimla", "state": "Himachal Pradesh", "latitude": 31.1048, "longitude": 77.1734, "zone": "Western Himalayan"},
    {"district": "Mandi", "state": "Himachal Pradesh", "latitude": 31.5892, "longitude": 76.9182, "zone": "Western Himalayan"},
    {"district": "Kangra", "state": "Himachal Pradesh", "latitude": 32.0998, "longitude": 76.2691, "zone": "Western Himalayan"},
    {"district": "Dehradun", "state": "Uttarakhand", "latitude": 30.3165, "longitude": 78.0322, "zone": "Western Himalayan"},
    {"district": "Pantnagar", "state": "Uttarakhand", "latitude": 29.0208, "longitude": 79.4897, "zone": "Western Himalayan"},
    {"district": "Gangtok", "state": "Sikkim", "latitude": 27.3389, "longitude": 88.6065, "zone": "Eastern Himalayan"},

    # --- EAST ZONE ---
    {"district": "Patna", "state": "Bihar", "latitude": 25.5941, "longitude": 85.1376, "zone": "Middle Gangetic"},
    {"district": "Gaya", "state": "Bihar", "latitude": 24.7914, "longitude": 85.0002, "zone": "Middle Gangetic"},
    {"district": "Muzaffarpur", "state": "Bihar", "latitude": 26.1209, "longitude": 85.3647, "zone": "Middle Gangetic"},
    {"district": "Bhagalpur", "state": "Bihar", "latitude": 25.2425, "longitude": 86.9842, "zone": "Middle Gangetic"},
    {"district": "Ranchi", "state": "Jharkhand", "latitude": 23.3441, "longitude": 85.3096, "zone": "Eastern Plateau"},
    {"district": "Dhanbad", "state": "Jharkhand", "latitude": 23.7957, "longitude": 86.4304, "zone": "Eastern Plateau"},
    {"district": "Dumka", "state": "Jharkhand", "latitude": 24.2677, "longitude": 87.2483, "zone": "Eastern Plateau"},
    {"district": "Bardhaman", "state": "West Bengal", "latitude": 23.2324, "longitude": 87.8615, "zone": "Lower Gangetic"},
    {"district": "Murshidabad", "state": "West Bengal", "latitude": 24.1759, "longitude": 88.2802, "zone": "Lower Gangetic"},
    {"district": "Jalpaiguri", "state": "West Bengal", "latitude": 26.5405, "longitude": 88.7196, "zone": "Eastern Himalayan"},
    {"district": "Midnapore", "state": "West Bengal", "latitude": 22.4257, "longitude": 87.3199, "zone": "Lower Gangetic"},
    {"district": "Cuttack", "state": "Odisha", "latitude": 20.4625, "longitude": 85.8828, "zone": "East Coast Plains"},
    {"district": "Sambalpur", "state": "Odisha", "latitude": 21.4669, "longitude": 83.9812, "zone": "Eastern Plateau"},
    {"district": "Balasore", "state": "Odisha", "latitude": 21.4934, "longitude": 86.9135, "zone": "East Coast Plains"},
    {"district": "Koraput", "state": "Odisha", "latitude": 18.8135, "longitude": 82.7123, "zone": "Eastern Ghats"},

    # --- CENTRAL ZONE ---
    {"district": "Bhopal", "state": "Madhya Pradesh", "latitude": 23.2599, "longitude": 77.4126, "zone": "Central Plateau"},
    {"district": "Indore", "state": "Madhya Pradesh", "latitude": 22.7196, "longitude": 75.8577, "zone": "Central Plateau"},
    {"district": "Jabalpur", "state": "Madhya Pradesh", "latitude": 23.1815, "longitude": 79.9864, "zone": "Central Plateau"},
    {"district": "Ujjain", "state": "Madhya Pradesh", "latitude": 23.1765, "longitude": 75.7885, "zone": "Central Plateau"},
    {"district": "Gwalior", "state": "Madhya Pradesh", "latitude": 26.2183, "longitude": 78.1828, "zone": "Central Plateau"},
    {"district": "Hoshangabad", "state": "Madhya Pradesh", "latitude": 22.7519, "longitude": 77.7289, "zone": "Central Plateau"},
    {"district": "Raipur", "state": "Chhattisgarh", "latitude": 21.2514, "longitude": 81.6296, "zone": "Eastern Plateau"},
    {"district": "Bilaspur", "state": "Chhattisgarh", "latitude": 22.0797, "longitude": 82.1409, "zone": "Eastern Plateau"},
    {"district": "Bastar", "state": "Chhattisgarh", "latitude": 19.1071, "longitude": 81.9535, "zone": "Eastern Plateau"},

    # --- WEST & ARID ZONE ---
    {"district": "Jaipur", "state": "Rajasthan", "latitude": 26.9124, "longitude": 75.7873, "zone": "Central Dry"},
    {"district": "Jodhpur", "state": "Rajasthan", "latitude": 26.2389, "longitude": 73.0243, "zone": "Western Dry"},
    {"district": "Bikaner", "state": "Rajasthan", "latitude": 28.0229, "longitude": 73.3119, "zone": "Western Dry"},
    {"district": "Barmer", "state": "Rajasthan", "latitude": 25.7521, "longitude": 71.3967, "zone": "Western Dry"},
    {"district": "Kota", "state": "Rajasthan", "latitude": 25.2138, "longitude": 75.8648, "zone": "Central Plateau"},
    {"district": "Sri Ganganagar", "state": "Rajasthan", "latitude": 29.9038, "longitude": 73.8772, "zone": "Trans-Gangetic Arid"},
    {"district": "Ahmedabad", "state": "Gujarat", "latitude": 23.0225, "longitude": 72.5714, "zone": "Gujarat Plains"},
    {"district": "Anand", "state": "Gujarat", "latitude": 22.5645, "longitude": 72.9289, "zone": "Gujarat Plains"},
    {"district": "Rajkot", "state": "Gujarat", "latitude": 22.3039, "longitude": 70.8022, "zone": "Saurashtra Arid"},
    {"district": "Junagadh", "state": "Gujarat", "latitude": 21.5222, "longitude": 70.4579, "zone": "Saurashtra Coastal"},
    {"district": "Surat", "state": "Gujarat", "latitude": 21.1702, "longitude": 72.8311, "zone": "Gujarat Coastal"},
    {"district": "Bhuj", "state": "Gujarat", "latitude": 23.2420, "longitude": 69.6669, "zone": "Kutch Arid"},

    # --- MAHARASHTRA & DECCAN ---
    {"district": "Pune", "state": "Maharashtra", "latitude": 18.5204, "longitude": 73.8567, "zone": "Western Plateau"},
    {"district": "Nashik", "state": "Maharashtra", "latitude": 19.9975, "longitude": 73.7898, "zone": "Western Plateau"},
    {"district": "Nagpur", "state": "Maharashtra", "latitude": 21.1458, "longitude": 79.0882, "zone": "Vidarbha"},
    {"district": "Akola", "state": "Maharashtra", "latitude": 20.7002, "longitude": 77.0082, "zone": "Vidarbha"},
    {"district": "Chhatrapati Sambhajinagar", "state": "Maharashtra", "latitude": 19.8762, "longitude": 75.3433, "zone": "Marathwada"},
    {"district": "Solapur", "state": "Maharashtra", "latitude": 17.6599, "longitude": 75.9064, "zone": "Western Plateau Semi-Arid"},
    {"district": "Kolhapur", "state": "Maharashtra", "latitude": 16.7050, "longitude": 74.2433, "zone": "Western Ghats"},
    {"district": "Ratnagiri", "state": "Maharashtra", "latitude": 16.9902, "longitude": 73.3120, "zone": "West Coast Plains"},

    # --- SOUTH ZONE ---
    {"district": "Kurnool", "state": "Andhra Pradesh", "latitude": 15.8281, "longitude": 78.0373, "zone": "Rayalaseema Semi-Arid"},
    {"district": "Anantapur", "state": "Andhra Pradesh", "latitude": 14.6819, "longitude": 77.6006, "zone": "Rayalaseema Arid"},
    {"district": "Guntur", "state": "Andhra Pradesh", "latitude": 16.3067, "longitude": 80.4365, "zone": "East Coast Plains"},
    {"district": "East Godavari", "state": "Andhra Pradesh", "latitude": 17.0005, "longitude": 81.8040, "zone": "East Coast Plains"},
    {"district": "Visakhapatnam", "state": "Andhra Pradesh", "latitude": 17.6868, "longitude": 83.2185, "zone": "East Coast Plains"},
    {"district": "Warangal", "state": "Telangana", "latitude": 17.9689, "longitude": 79.5941, "zone": "Southern Plateau"},
    {"district": "Nizamabad", "state": "Telangana", "latitude": 18.6725, "longitude": 78.0941, "zone": "Southern Plateau"},
    {"district": "Nalgonda", "state": "Telangana", "latitude": 17.0575, "longitude": 79.2684, "zone": "Southern Plateau"},
    {"district": "Bengaluru Rural", "state": "Karnataka", "latitude": 13.2847, "longitude": 77.5583, "zone": "Southern Plateau"},
    {"district": "Belagavi", "state": "Karnataka", "latitude": 15.8497, "longitude": 74.4977, "zone": "Northern Dry Zone"},
    {"district": "Dharwad", "state": "Karnataka", "latitude": 15.4589, "longitude": 75.0078, "zone": "Northern Transition Zone"},
    {"district": "Kalaburagi", "state": "Karnataka", "latitude": 17.3297, "longitude": 76.8343, "zone": "Northern Dry Zone"},
    {"district": "Ballari", "state": "Karnataka", "latitude": 15.1394, "longitude": 76.9214, "zone": "Northern Dry Zone"},
    {"district": "Mysuru", "state": "Karnataka", "latitude": 12.2958, "longitude": 76.6394, "zone": "Southern Dry Zone"},
    {"district": "Shivamogga", "state": "Karnataka", "latitude": 13.9299, "longitude": 75.5681, "zone": "Hilly / Malnad"},
    {"district": "Thanjavur", "state": "Tamil Nadu", "latitude": 10.7870, "longitude": 79.1378, "zone": "Cauvery Delta"},
    {"district": "Coimbatore", "state": "Tamil Nadu", "latitude": 11.0168, "longitude": 76.9558, "zone": "Western Zone"},
    {"district": "Madurai", "state": "Tamil Nadu", "latitude": 9.9252, "longitude": 78.1198, "zone": "Southern Zone"},
    {"district": "Tiruchirappalli", "state": "Tamil Nadu", "latitude": 10.7905, "longitude": 78.7047, "zone": "Cauvery Delta"},
    {"district": "Tirunelveli", "state": "Tamil Nadu", "latitude": 8.7139, "longitude": 77.7567, "zone": "Southern Coastal"},
    {"district": "Palakkad", "state": "Kerala", "latitude": 10.7867, "longitude": 76.6548, "zone": "Palakkad Plains"},
    {"district": "Alappuzha", "state": "Kerala", "latitude": 9.4981, "longitude": 76.3388, "zone": "Kuttanad Wet Zone"},
    {"district": "Wayanad", "state": "Kerala", "latitude": 11.6854, "longitude": 76.1320, "zone": "Highland / Western Ghats"},
    {"district": "Kozhikode", "state": "Kerala", "latitude": 11.2588, "longitude": 75.7804, "zone": "West Coast Plains"},

    # --- NORTHEAST ZONE ---
    {"district": "Kamrup", "state": "Assam", "latitude": 26.2006, "longitude": 91.5606, "zone": "Brahmaputra Valley"},
    {"district": "Jorhat", "state": "Assam", "latitude": 26.7509, "longitude": 94.2037, "zone": "Upper Brahmaputra"},
    {"district": "Cachar", "state": "Assam", "latitude": 24.8333, "longitude": 92.7789, "zone": "Barak Valley"},
    {"district": "Shillong", "state": "Meghalaya", "latitude": 25.5788, "longitude": 91.8933, "zone": "Eastern Himalayan"},
    {"district": "Dimapur", "state": "Nagaland", "latitude": 25.9068, "longitude": 93.7274, "zone": "Eastern Himalayan"},
    {"district": "Imphal West", "state": "Manipur", "latitude": 24.8170, "longitude": 93.9368, "zone": "Eastern Himalayan"},
    {"district": "Agartala", "state": "Tripura", "latitude": 23.8315, "longitude": 91.2868, "zone": "Eastern Himalayan"},
    {"district": "Aizawl", "state": "Mizoram", "latitude": 23.7271, "longitude": 92.7176, "zone": "Eastern Himalayan"},
    {"district": "Papum Pare", "state": "Arunachal Pradesh", "latitude": 27.1020, "longitude": 93.6920, "zone": "Eastern Himalayan"}
]

locations_df = pd.DataFrame(PAN_INDIA_DISTRICTS)
print(f"Total Pan-India Agro-Climatic Nodes Registered: {len(locations_df)}")
print(f"Total States/UTs Covered: {locations_df['state'].nunique()}")
print(f"Total Agro-Climatic Sub-zones Represented: {locations_df['zone'].nunique()}")

Total Pan-India Agro-Climatic Nodes Registered: 99
Total States/UTs Covered: 28
Total Agro-Climatic Sub-zones Represented: 41


In [ ]:
# ==============================================================================
# BLOCK 2: ROBUST & STABLE PAN-INDIA INGESTION ENGINE
# ==============================================================================
!pip install -q openmeteo-requests requests_cache retry_requests pyarrow

import openmeteo_requests
import requests_cache
import pandas as pd
import numpy as np
import time
from retry_requests import retry
from tqdm.auto import tqdm

# 1. Setup cached session with exponential retry backoff
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.3)
openmeteo = openmeteo_requests.Client(session=retry_session)

URL = "https://archive-api.open-meteo.com/v1/archive"

# 2. Selected Core Variables matching Step 1 requirements
DAILY_PARAMS = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "precipitation_sum",
    "rain_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "wind_direction_10m_dominant",
    "surface_pressure_mean",
    "cloud_cover_mean",
    "dew_point_2m_mean",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
    "soil_temperature_0_to_7cm_mean",
    "soil_moisture_0_to_7cm_mean",
    "soil_moisture_7_to_28cm_mean"
]

all_district_dfs = []
failed_districts = []

START_DATE = "2020-01-01"
END_DATE = "2024-12-31"  # 5 full complete years (2020-2024)

print(f"Ingesting {len(locations_df)} districts from {START_DATE} to {END_DATE}...")

for idx, loc in enumerate(tqdm(locations_df.to_dict('records'), desc="Fetching Weather Records")):
    params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": DAILY_PARAMS,
        "timezone": "Asia/Kolkata"
    }

    try:
        responses = openmeteo.weather_api(URL, params=params)
        response = responses[0]

        # Extract daily attributes
        daily = response.Daily()
        dates = pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s", utc=True).tz_convert("Asia/Kolkata"),
            end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True).tz_convert("Asia/Kolkata"),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        )

        data_dict = {"date": dates.date}

        for i, var_name in enumerate(DAILY_PARAMS):
            data_dict[var_name] = daily.Variables(i).ValuesAsNumpy()

        loc_df = pd.DataFrame(data_dict)

        # Attach geospatial labels
        loc_df["district"] = loc["district"]
        loc_df["state"] = loc["state"]
        loc_df["zone"] = loc["zone"]
        loc_df["latitude"] = loc["latitude"]
        loc_df["longitude"] = loc["longitude"]

        all_district_dfs.append(loc_df)

    except Exception as e:
        failed_districts.append({"district": loc["district"], "error": str(e)})
        time.sleep(1.0)

# Check if data was retrieved
if not all_district_dfs:
    raise RuntimeError(f"All requests failed. First error: {failed_districts[0] if failed_districts else 'Unknown'}")

if failed_districts:
    print(f"Warning: {len(failed_districts)} districts failed out of {len(locations_df)}.")

# 3. Concatenate and serialize
df_india_raw = pd.concat(all_district_dfs, ignore_index=True)

# Reorder columns
meta_cols = ["date", "district", "state", "zone", "latitude", "longitude"]
metric_cols = [c for c in df_india_raw.columns if c not in meta_cols]
df_india_raw = df_india_raw[meta_cols + metric_cols]

# Sort by district and date
df_india_raw["date"] = pd.to_datetime(df_india_raw["date"])
df_india_raw = df_india_raw.sort_values(by=["district", "date"]).reset_index(drop=True)

# Save to Parquet
output_path = "pan_india_farmer_weather_raw.parquet"
df_india_raw.to_parquet(output_path, engine="pyarrow", index=False)

print("\n" + "="*50)
print(f"SUCCESS: Pipeline Ingested Clean Dataset!")
print(f"Total Rows Extracted: {len(df_india_raw):,}")
print(f"Total Columns: {df_india_raw.shape[1]}")
print(f"Saved locally to: {output_path}")
print("="*50)

Ingesting 99 districts from 2020-01-01 to 2024-12-31...


Fetching Weather Records:   0%|          | 0/99 [00:00<?, ?it/s]


SUCCESS: Pipeline Ingested Clean Dataset!
Total Rows Extracted: 32,886
Total Columns: 23
Saved locally to: pan_india_farmer_weather_raw.parquet


In [ ]:
!pip install --upgrade openmeteo-requests requests-cache retry-requests pyarrow fastparquet tqdm

In [ ]:
# ==============================================================================
# STEP 3: NATIONWIDE 75-DISTRICT CONCURRENT HARVESTER (NASA POWER API)
# ==============================================================================
import time
import requests
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# 75 Agricultural Anchors covering all 28 States, 8 UTs & Major Sub-zones
ALL_INDIA_75_DISTRICTS = [
    # --- NORTH ZONE ---
    {"district": "Ludhiana", "state": "Punjab", "latitude": 30.9010, "longitude": 75.8573, "zone": "Trans-Gangetic"},
    {"district": "Bathinda", "state": "Punjab", "latitude": 30.2110, "longitude": 74.9455, "zone": "Trans-Gangetic"},
    {"district": "Amritsar", "state": "Punjab", "latitude": 31.6340, "longitude": 74.8723, "zone": "Trans-Gangetic"},
    {"district": "Karnal", "state": "Haryana", "latitude": 29.6857, "longitude": 76.9905, "zone": "Trans-Gangetic"},
    {"district": "Hisar", "state": "Haryana", "latitude": 29.1492, "longitude": 75.7217, "zone": "Trans-Gangetic"},
    {"district": "Sirsa", "state": "Haryana", "latitude": 29.5349, "longitude": 75.0298, "zone": "Trans-Gangetic"},
    {"district": "Meerut", "state": "Uttar Pradesh", "latitude": 28.9845, "longitude": 77.7064, "zone": "Upper Gangetic"},
    {"district": "Agra", "state": "Uttar Pradesh", "latitude": 27.1767, "longitude": 78.0081, "zone": "Upper Gangetic"},
    {"district": "Varanasi", "state": "Uttar Pradesh", "latitude": 25.3176, "longitude": 82.9739, "zone": "Middle Gangetic"},
    {"district": "Gorakhpur", "state": "Uttar Pradesh", "latitude": 26.7606, "longitude": 83.3732, "zone": "Middle Gangetic"},
    {"district": "Jhansi", "state": "Uttar Pradesh", "latitude": 25.4484, "longitude": 78.5685, "zone": "Central Plateau"},
    {"district": "Bareilly", "state": "Uttar Pradesh", "latitude": 28.3670, "longitude": 79.4304, "zone": "Upper Gangetic"},
    {"district": "New Delhi", "state": "Delhi", "latitude": 28.6139, "longitude": 77.2090, "zone": "Upper Gangetic"},
    {"district": "Chandigarh", "state": "Chandigarh", "latitude": 30.7333, "longitude": 76.7794, "zone": "Trans-Gangetic"},

    # --- HIMALAYAN & NORTHERN MOUNTAINS ---
    {"district": "Srinagar", "state": "Jammu and Kashmir", "latitude": 34.0837, "longitude": 74.7973, "zone": "Western Himalayan"},
    {"district": "Jammu", "state": "Jammu and Kashmir", "latitude": 32.7266, "longitude": 74.8570, "zone": "Western Himalayan"},
    {"district": "Leh", "state": "Ladakh", "latitude": 34.1526, "longitude": 77.5771, "zone": "Cold Arid Himalayan"},
    {"district": "Kargil", "state": "Ladakh", "latitude": 34.5539, "longitude": 76.1349, "zone": "Cold Arid Himalayan"},
    {"district": "Shimla", "state": "Himachal Pradesh", "latitude": 31.1048, "longitude": 77.1734, "zone": "Western Himalayan"},
    {"district": "Mandi", "state": "Himachal Pradesh", "latitude": 31.5892, "longitude": 76.9182, "zone": "Western Himalayan"},
    {"district": "Kangra", "state": "Himachal Pradesh", "latitude": 32.0998, "longitude": 76.2691, "zone": "Western Himalayan"},
    {"district": "Dehradun", "state": "Uttarakhand", "latitude": 30.3165, "longitude": 78.0322, "zone": "Western Himalayan"},
    {"district": "Pantnagar", "state": "Uttarakhand", "latitude": 29.0208, "longitude": 79.4897, "zone": "Tarai Zone"},

    # --- EAST ZONE ---
    {"district": "Patna", "state": "Bihar", "latitude": 25.5941, "longitude": 85.1376, "zone": "Middle Gangetic"},
    {"district": "Gaya", "state": "Bihar", "latitude": 24.7914, "longitude": 85.0002, "zone": "Middle Gangetic"},
    {"district": "Muzaffarpur", "state": "Bihar", "latitude": 26.1209, "longitude": 85.3647, "zone": "Middle Gangetic"},
    {"district": "Ranchi", "state": "Jharkhand", "latitude": 23.3441, "longitude": 85.3096, "zone": "Eastern Plateau"},
    {"district": "Dhanbad", "state": "Jharkhand", "latitude": 23.7957, "longitude": 86.4304, "zone": "Eastern Plateau"},
    {"district": "Bardhaman", "state": "West Bengal", "latitude": 23.2324, "longitude": 87.8615, "zone": "Lower Gangetic"},
    {"district": "Jalpaiguri", "state": "West Bengal", "latitude": 26.5405, "longitude": 88.7196, "zone": "Eastern Himalayan"},
    {"district": "Midnapore", "state": "West Bengal", "latitude": 22.4257, "longitude": 87.3199, "zone": "Lower Gangetic"},
    {"district": "Cuttack", "state": "Odisha", "latitude": 20.4625, "longitude": 85.8828, "zone": "East Coast Plains"},
    {"district": "Sambalpur", "state": "Odisha", "latitude": 21.4669, "longitude": 83.9812, "zone": "Eastern Plateau"},
    {"district": "Koraput", "state": "Odisha", "latitude": 18.8135, "longitude": 82.7123, "zone": "Eastern Ghats"},
    {"district": "Gangtok", "state": "Sikkim", "latitude": 27.3389, "longitude": 88.6065, "zone": "Eastern Himalayan"},

    # --- CENTRAL ZONE ---
    {"district": "Bhopal", "state": "Madhya Pradesh", "latitude": 23.2599, "longitude": 77.4126, "zone": "Central Plateau"},
    {"district": "Indore", "state": "Madhya Pradesh", "latitude": 22.7196, "longitude": 75.8577, "zone": "Malwa Plateau"},
    {"district": "Jabalpur", "state": "Madhya Pradesh", "latitude": 23.1815, "longitude": 79.9864, "zone": "Central Plateau"},
    {"district": "Gwalior", "state": "Madhya Pradesh", "latitude": 26.2183, "longitude": 78.1828, "zone": "Gird Zone"},
    {"district": "Hoshangabad", "state": "Madhya Pradesh", "latitude": 22.7519, "longitude": 77.7289, "zone": "Narmada Valley"},
    {"district": "Raipur", "state": "Chhattisgarh", "latitude": 21.2514, "longitude": 81.6296, "zone": "Eastern Plateau"},
    {"district": "Bilaspur", "state": "Chhattisgarh", "latitude": 22.0797, "longitude": 82.1409, "zone": "Eastern Plateau"},
    {"district": "Bastar", "state": "Chhattisgarh", "latitude": 19.1071, "longitude": 81.9535, "zone": "Bastar Plateau"},

    # --- WEST & ARID ZONE ---
    {"district": "Jaipur", "state": "Rajasthan", "latitude": 26.9124, "longitude": 75.7873, "zone": "Central Dry"},
    {"district": "Jodhpur", "state": "Rajasthan", "latitude": 26.2389, "longitude": 73.0243, "zone": "Western Dry"},
    {"district": "Barmer", "state": "Rajasthan", "latitude": 25.7521, "longitude": 71.3967, "zone": "Western Arid"},
    {"district": "Bikaner", "state": "Rajasthan", "latitude": 28.0229, "longitude": 73.3119, "zone": "Hyper-Arid"},
    {"district": "Kota", "state": "Rajasthan", "latitude": 25.2138, "longitude": 75.8648, "zone": "Hadoti Plateau"},
    {"district": "Sri Ganganagar", "state": "Rajasthan", "latitude": 29.9038, "longitude": 73.8772, "zone": "Trans-Gangetic Arid"},
    {"district": "Ahmedabad", "state": "Gujarat", "latitude": 23.0225, "longitude": 72.5714, "zone": "Gujarat Plains"},
    {"district": "Anand", "state": "Gujarat", "latitude": 22.5645, "longitude": 72.9289, "zone": "Middle Gujarat"},
    {"district": "Rajkot", "state": "Gujarat", "latitude": 22.3039, "longitude": 70.8022, "zone": "Saurashtra Dry"},
    {"district": "Junagadh", "state": "Gujarat", "latitude": 21.5222, "longitude": 70.4579, "zone": "South Saurashtra Coastal"},
    {"district": "Bhuj", "state": "Gujarat", "latitude": 23.2420, "longitude": 69.6669, "zone": "Kutch Arid"},

    # --- MAHARASHTRA & GOA ---
    {"district": "Pune", "state": "Maharashtra", "latitude": 18.5204, "longitude": 73.8567, "zone": "Western Plateau"},
    {"district": "Nashik", "state": "Maharashtra", "latitude": 19.9975, "longitude": 73.7898, "zone": "Western Ghats Transition"},
    {"district": "Nagpur", "state": "Maharashtra", "latitude": 21.1458, "longitude": 79.0882, "zone": "Vidarbha"},
    {"district": "Akola", "state": "Maharashtra", "latitude": 20.7002, "longitude": 77.0082, "zone": "Vidarbha Semi-Arid"},
    {"district": "Chhatrapati Sambhajinagar", "state": "Maharashtra", "latitude": 19.8762, "longitude": 75.3433, "zone": "Marathwada"},
    {"district": "Solapur", "state": "Maharashtra", "latitude": 17.6599, "longitude": 75.9064, "zone": "Scarcity Zone"},
    {"district": "Kolhapur", "state": "Maharashtra", "latitude": 16.7050, "longitude": 74.2433, "zone": "Sub-Montane"},
    {"district": "Ratnagiri", "state": "Maharashtra", "latitude": 16.9902, "longitude": 73.3120, "zone": "Konkan Coastal"},
    {"district": "North Goa", "state": "Goa", "latitude": 15.4989, "longitude": 73.8278, "zone": "West Coast Wet"},

    # --- SOUTH ZONE ---
    {"district": "Guntur", "state": "Andhra Pradesh", "latitude": 16.3067, "longitude": 80.4365, "zone": "Krishna-Godavari Coastal"},
    {"district": "Anantapur", "state": "Andhra Pradesh", "latitude": 14.6819, "longitude": 77.6006, "zone": "Rayalaseema Scarce"},
    {"district": "Kurnool", "state": "Andhra Pradesh", "latitude": 15.8281, "longitude": 78.0373, "zone": "Scarce Rainfall Zone"},
    {"district": "Visakhapatnam", "state": "Andhra Pradesh", "latitude": 17.6868, "longitude": 83.2185, "zone": "North Coastal"},
    {"district": "Warangal", "state": "Telangana", "latitude": 17.9689, "longitude": 79.5941, "zone": "Central Telangana"},
    {"district": "Nizamabad", "state": "Telangana", "latitude": 18.6725, "longitude": 78.0941, "zone": "Northern Telangana"},
    {"district": "Nalgonda", "state": "Telangana", "latitude": 17.0575, "longitude": 79.2684, "zone": "Southern Telangana"},
    {"district": "Dharwad", "state": "Karnataka", "latitude": 15.4589, "longitude": 75.0078, "zone": "Northern Transition"},
    {"district": "Belagavi", "state": "Karnataka", "latitude": 15.8497, "longitude": 74.4977, "zone": "Northern Dry Zone"},
    {"district": "Ballari", "state": "Karnataka", "latitude": 15.1394, "longitude": 76.9214, "zone": "North-East Dry Zone"},
    {"district": "Mysuru", "state": "Karnataka", "latitude": 12.2958, "longitude": 76.6394, "zone": "Southern Dry Zone"},
    {"district": "Shivamogga", "state": "Karnataka", "latitude": 13.9299, "longitude": 75.5681, "zone": "Hilly Malnad"},
    {"district": "Coimbatore", "state": "Tamil Nadu", "latitude": 11.0168, "longitude": 76.9558, "zone": "Western Zone"},
    {"district": "Thanjavur", "state": "Tamil Nadu", "latitude": 10.7870, "longitude": 79.1378, "zone": "Cauvery Delta"},
    {"district": "Madurai", "state": "Tamil Nadu", "latitude": 9.9252, "longitude": 78.1198, "zone": "Southern Zone"},
    {"district": "Tirunelveli", "state": "Tamil Nadu", "latitude": 8.7139, "longitude": 77.7567, "zone": "Southern Coastal"},
    {"district": "Palakkad", "state": "Kerala", "latitude": 10.7867, "longitude": 76.6548, "zone": "Palakkad Plains"},
    {"district": "Alappuzha", "state": "Kerala", "latitude": 9.4981, "longitude": 76.3388, "zone": "Kuttanad Wet Zone"},
    {"district": "Wayanad", "state": "Kerala", "latitude": 11.6854, "longitude": 76.1320, "zone": "Highland Ghats"},
    {"district": "Puducherry", "state": "Puducherry", "latitude": 11.9416, "longitude": 79.8083, "zone": "Coromandel Coastal"},

    # --- NORTHEAST ZONE (ALL 8 STATES) ---
    {"district": "Kamrup", "state": "Assam", "latitude": 26.2006, "longitude": 91.5606, "zone": "Brahmaputra Valley"},
    {"district": "Jorhat", "state": "Assam", "latitude": 26.7509, "longitude": 94.2037, "zone": "Upper Brahmaputra"},
    {"district": "Cachar", "state": "Assam", "latitude": 24.8333, "longitude": 92.7789, "zone": "Barak Valley"},
    {"district": "Shillong", "state": "Meghalaya", "latitude": 25.5788, "longitude": 91.8933, "zone": "Eastern Himalayan"},
    {"district": "Papum Pare", "state": "Arunachal Pradesh", "latitude": 27.1020, "longitude": 93.6920, "zone": "Arunachal Hills"},
    {"district": "Dimapur", "state": "Nagaland", "latitude": 25.9068, "longitude": 93.7274, "zone": "Nagaland Foothills"},
    {"district": "Imphal West", "state": "Manipur", "latitude": 24.8170, "longitude": 93.9368, "zone": "Manipur Valley"},
    {"district": "Aizawl", "state": "Mizoram", "latitude": 23.7271, "longitude": 92.7176, "zone": "Mizoram Hills"},
    {"district": "Agartala", "state": "Tripura", "latitude": 23.8315, "longitude": 91.2868, "zone": "Tripura Plains"},

    # --- UNION TERRITORIES & ISLANDS ---
    {"district": "Daman", "state": "Dadra and Nagar Haveli and Daman and Diu", "latitude": 20.3974, "longitude": 72.8328, "zone": "West Coast Littoral"},
    {"district": "Port Blair", "state": "Andaman and Nicobar Islands", "latitude": 11.6234, "longitude": 92.7265, "zone": "Island Tropical Wet"},
    {"district": "Kavaratti", "state": "Lakshadweep", "latitude": 10.5669, "longitude": 72.6420, "zone": "Island Marine"}
]

NASA_POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMETERS = "T2M_MAX,T2M_MIN,T2M,RH2M,PRECTOTCORR,WS10M,WS10M_MAX,PS,ALLSKY_SFC_SW_DWN,T2MDEW"
START_DATE = "20210101"
END_DATE = "20241231"

def fetch_district_data(loc):
    """Safe retrieval worker with session persistence and auto-retry."""
    params = {
        "parameters": PARAMETERS,
        "community": "AG",
        "longitude": loc["longitude"],
        "latitude": loc["latitude"],
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }

    sess = requests.Session()
    for attempt in range(4):
        try:
            resp = sess.get(NASA_POWER_URL, params=params, timeout=45)
            if resp.status_code == 200:
                payload = resp.json()
                raw_param_data = payload["properties"]["parameter"]

                df_sub = pd.DataFrame(raw_param_data)
                df_sub.index.name = "date_str"
                df_sub = df_sub.reset_index()

                df_sub["date"] = pd.to_datetime(df_sub["date_str"], format="%Y%m%d")
                df_sub.drop(columns=["date_str"], inplace=True)

                df_sub["district"] = loc["district"]
                df_sub["state"] = loc["state"]
                df_sub["zone"] = loc["zone"]
                df_sub["latitude"] = loc["latitude"]
                df_sub["longitude"] = loc["longitude"]
                return df_sub
            else:
                time.sleep(1.5 * (attempt + 1))
        except Exception:
            time.sleep(2.0)
    return None

print(f"Launching multi-threaded harvester for {len(ALL_INDIA_75_DISTRICTS)} Pan-India districts...")
collected_dfs = []

# 6 concurrent threads achieve the highest throughput on NASA POWER without network throttles
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {executor.submit(fetch_district_data, loc): loc for loc in ALL_INDIA_75_DISTRICTS}
    for f in tqdm(as_completed(futures), total=len(futures), desc="Extracting All-India Weather"):
        res = f.result()
        if res is not None:
            collected_dfs.append(res)
        time.sleep(0.15)

if not collected_dfs:
    raise RuntimeError("Harvesting failed. Verify internet connection.")

df_raw = pd.concat(collected_dfs, ignore_index=True)

# Standardize meteorological column names
rename_dict = {
    "T2M": "temperature_2m_mean",
    "T2M_MIN": "temperature_2m_min",
    "T2M_MAX": "temperature_2m_max",
    "RH2M": "relative_humidity_2m_mean",
    "PRECTOTCORR": "precipitation_sum",
    "WS10M": "wind_speed_10m_mean",
    "WS10M_MAX": "wind_speed_10m_max",
    "PS": "surface_pressure_mean",
    "ALLSKY_SFC_SW_DWN": "shortwave_radiation_sum",
    "T2MDEW": "dew_point_2m_mean"
}
df_raw.rename(columns=rename_dict, inplace=True)
df_raw.replace(-999.0, np.nan, inplace=True)

# Standardize wind units (m/s to km/h) for CIBRC spray benchmarks
df_raw["wind_speed_10m_max"] = df_raw["wind_speed_10m_max"] * 3.6
df_raw["wind_speed_10m_mean"] = df_raw["wind_speed_10m_mean"] * 3.6

lead_cols = ["date", "district", "state", "zone", "latitude", "longitude"]
tail_cols = [c for c in df_raw.columns if c not in lead_cols]
df_raw = df_raw[lead_cols + tail_cols].sort_values(by=["district", "date"]).reset_index(drop=True)

output_parquet = "pan_india_farmer_weather_raw.parquet"
df_raw.to_parquet(output_parquet, engine="pyarrow", index=False)

print("\n" + "=" * 60)
print("SUCCESS: 75-District Pan-India Agroclimatology Dataset Extracted!")
print(f"Total Rows Extracted : {len(df_raw):,}")
print(f"Total Districts      : {df_raw['district'].nunique()}")
print(f"Total States & UTs   : {df_raw['state'].nunique()}")
print(f"Date Span            : {df_raw['date'].min().date()} to {df_raw['date'].max().date()}")
print(f"Saved to Disk        : {output_parquet}")
print("=" * 60)

Launching multi-threaded harvester for 95 Pan-India districts...


Extracting All-India Weather:   0%|          | 0/95 [00:00<?, ?it/s]


SUCCESS: 75-District Pan-India Agroclimatology Dataset Extracted!
Total Rows Extracted : 138,795
Total Districts      : 95
Total States & UTs   : 36
Date Span            : 2021-01-01 to 2024-12-31
Saved to Disk        : pan_india_farmer_weather_raw.parquet


In [ ]:
df_raw.head()

,date,district,state,zone,latitude,longitude,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_mean,wind_speed_10m_max,surface_pressure_mean,shortwave_radiation_sum,dew_point_2m_mean
0,2021-01-01,Agartala,Tripura,Tripura Plains,23.8315,91.2868,23.93,12.84,18.05,78.22,0.0,4.068,7.524,101.22,16.04,13.93
1,2021-01-02,Agartala,Tripura,Tripura Plains,23.8315,91.2868,23.71,12.71,17.95,79.44,0.0,4.680,7.704,101.19,16.89,14.10
2,2021-01-03,Agartala,Tripura,Tripura Plains,23.8315,91.2868,23.98,13.48,18.26,77.43,0.0,4.284,7.308,101.03,16.66,13.98
3,2021-01-04,Agartala,Tripura,Tripura Plains,23.8315,91.2868,24.71,12.95,18.54,77.73,0.0,4.968,6.048,101.08,16.50,14.33
4,2021-01-05,Agartala,Tripura,Tripura Plains,23.8315,91.2868,25.04,13.61,18.96,79.11,0.0,5.724,7.416,101.08,16.41,15.01


In [ ]:
# ==============================================================================
# STEP 4 & 5: INSPECTION, CLEANING & PREPROCESSING
# ==============================================================================
import pandas as pd
import numpy as np

print("Loading raw extracted parquet...")
df_raw = pd.read_parquet("pan_india_farmer_weather_raw.parquet")

# 1. Structural Inspection
print(f"Shape: {df_raw.shape}")
print(f"Duplicates: {df_raw.duplicated(subset=['district', 'date']).sum()}")

# Drop exact coordinate-date duplicates if any
df_clean = df_raw.drop_duplicates(subset=["district", "date"]).copy()
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean = df_clean.sort_values(by=["district", "date"]).reset_index(drop=True)

# 2. Quality Check: Replace physical anomalies
# NASA occasionally marks missing points with -999.0 or negative solar radiation
df_clean.replace(-999.0, np.nan, inplace=True)
df_clean["shortwave_radiation_sum"] = df_clean["shortwave_radiation_sum"].clip(lower=0.0)
df_clean["precipitation_sum"] = df_clean["precipitation_sum"].clip(lower=0.0)
df_clean["relative_humidity_2m_mean"] = df_clean["relative_humidity_2m_mean"].clip(lower=1.0, upper=100.0)

# 3. Handle Missing Values without Leakage (Per-district time interpolation)
numeric_cols = [
    "temperature_2m_mean", "temperature_2m_min", "temperature_2m_max",
    "relative_humidity_2m_mean", "precipitation_sum", "wind_speed_10m_mean",
    "wind_speed_10m_max", "surface_pressure_mean", "shortwave_radiation_sum",
    "dew_point_2m_mean"
]

print("\nMissing values before cleaning:")
print(df_clean[numeric_cols].isnull().sum()[lambda x: x > 0])

df_clean[numeric_cols] = df_clean.groupby("district")[numeric_cols].transform(
    lambda grp: grp.interpolate(method="linear").bfill().ffill()
)

print("\nMissing values after per-district interpolation:", df_clean[numeric_cols].isnull().sum().sum())

Loading raw extracted parquet...
Shape: (138795, 16)
Duplicates: 0

Missing values before cleaning:
Series([], dtype: int64)

Missing values after per-district interpolation: 0


In [ ]:
# ==============================================================================
# STEP 6: AGRO-CLIMATIC FEATURE ENGINEERING
# ==============================================================================
import pandas as pd
import numpy as np

print("Engineering agrometeorological horizons and physical indices...")

# 1. Calendar & Indian Agro-Seasons
df_clean["year"] = df_clean["date"].dt.year
df_clean["month"] = df_clean["date"].dt.month
df_clean["day_of_year"] = df_clean["date"].dt.dayofyear
df_clean["doy_sin"] = np.sin(2 * np.pi * df_clean["day_of_year"] / 365.25)
df_clean["doy_cos"] = np.cos(2 * np.pi * df_clean["day_of_year"] / 365.25)

def map_indian_season(m):
    if m in [6, 7, 8, 9, 10]:
        return "KHARIF"
    elif m in [11, 12, 1, 2, 3]:
        return "RABI"
    return "ZAID"

df_clean["agro_season"] = df_clean["month"].apply(map_indian_season)

# 2. Vapor Pressure Deficit (VPD in kPa)
# Saturation vapor pressure: e_s = 0.61078 * exp((17.27 * T) / (T + 237.3))
# Actual vapor pressure: e_a = e_s * (RH / 100)
es = 0.61078 * np.exp((17.27 * df_clean["temperature_2m_mean"]) / (df_clean["temperature_2m_mean"] + 237.3))
ea = es * (df_clean["relative_humidity_2m_mean"] / 100.0)
df_clean["vapor_pressure_deficit"] = np.clip(es - ea, 0, None)

# 3. Hargreaves-Samani Reference Evapotranspiration (ET0 mm/day)
t_diff = np.clip(df_clean["temperature_2m_max"] - df_clean["temperature_2m_min"], 0.1, None)
df_clean["et0_fao_evapotranspiration"] = (
    0.0023 * (df_clean["temperature_2m_mean"] + 17.8) * np.sqrt(t_diff) * (df_clean["shortwave_radiation_sum"] * 0.408)
)
df_clean["et0_fao_evapotranspiration"] = df_clean["et0_fao_evapotranspiration"].clip(lower=0.5, upper=12.0)

# 4. District-Grouped Rolling Horizons (Prevents Cross-Location Leakage)
grp = df_clean.groupby("district")

# Precipitation Windows
df_clean["rainfall_1d"] = df_clean["precipitation_sum"]
df_clean["rainfall_3d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_clean["rainfall_7d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(7, min_periods=1).sum())
df_clean["rainfall_14d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(14, min_periods=1).sum())

# Temperature Horizons
df_clean["temperature_3d_mean"] = grp["temperature_2m_mean"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["temperature_7d_mean"] = grp["temperature_2m_mean"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_clean["temperature_3d_max"] = grp["temperature_2m_max"].transform(lambda x: x.rolling(3, min_periods=1).max())
df_clean["temperature_7d_max"] = grp["temperature_2m_max"].transform(lambda x: x.rolling(7, min_periods=1).max())
df_clean["temperature_3d_min"] = grp["temperature_2m_min"].transform(lambda x: x.rolling(3, min_periods=1).min())

# Humidity Horizons
df_clean["humidity_3d_mean"] = grp["relative_humidity_2m_mean"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["humidity_7d_mean"] = grp["relative_humidity_2m_mean"].transform(lambda x: x.rolling(7, min_periods=1).mean())

# Evapotranspiration Horizons
df_clean["et0_3d_sum"] = grp["et0_fao_evapotranspiration"].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_clean["et0_7d_sum"] = grp["et0_fao_evapotranspiration"].transform(lambda x: x.rolling(7, min_periods=1).sum())

# Wind Horizons
df_clean["wind_speed_3d_mean"] = grp["wind_speed_10m_max"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["wind_gust_3d_max"] = grp["wind_speed_10m_max"].transform(lambda x: x.rolling(3, min_periods=1).max())

# 5. Consecutive Dry & Wet Days
def calc_streaks(series, condition_fn):
    flags = condition_fn(series).astype(int)
    streaks, cur = [], 0
    for v in flags:
        cur = cur + 1 if v == 1 else 0
        streaks.append(cur)
    return pd.Series(streaks, index=series.index)

df_clean["consecutive_dry_days"] = grp["precipitation_sum"].transform(lambda s: calc_streaks(s, lambda x: x < 1.0))
df_clean["consecutive_rain_days"] = grp["precipitation_sum"].transform(lambda s: calc_streaks(s, lambda x: x >= 2.5))

df_clean.to_parquet("pan_india_featured.parquet", engine="pyarrow", index=False)
print("Features saved to pan_india_featured.parquet")

Engineering agrometeorological horizons and physical indices...
Features saved to pan_india_featured.parquet


In [ ]:
# ==============================================================================
# ZERO-DISK-DEPENDENCY END-TO-END PIPELINE: INGEST -> FEATURES -> TARGETS
# ==============================================================================
import time
import requests
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# 1. 28 Representative State & Zone Anchor Districts Across India
INDIAN_GRID_28 = [
    {"district": "Ludhiana", "state": "Punjab", "latitude": 30.9010, "longitude": 75.8573, "zone": "Trans-Gangetic"},
    {"district": "Karnal", "state": "Haryana", "latitude": 29.6857, "longitude": 76.9905, "zone": "Trans-Gangetic"},
    {"district": "Meerut", "state": "Uttar Pradesh", "latitude": 28.9845, "longitude": 77.7064, "zone": "Upper Gangetic"},
    {"district": "Varanasi", "state": "Uttar Pradesh", "latitude": 25.3176, "longitude": 82.9739, "zone": "Middle Gangetic"},
    {"district": "Shimla", "state": "Himachal Pradesh", "latitude": 31.1048, "longitude": 77.1734, "zone": "Western Himalayan"},
    {"district": "Dehradun", "state": "Uttarakhand", "latitude": 30.3165, "longitude": 78.0322, "zone": "Western Himalayan"},
    {"district": "Srinagar", "state": "Jammu & Kashmir", "latitude": 34.0837, "longitude": 74.7973, "zone": "Western Himalayan"},
    {"district": "Patna", "state": "Bihar", "latitude": 25.5941, "longitude": 85.1376, "zone": "Middle Gangetic"},
    {"district": "Ranchi", "state": "Jharkhand", "latitude": 23.3441, "longitude": 85.3096, "zone": "Eastern Plateau"},
    {"district": "Bardhaman", "state": "West Bengal", "latitude": 23.2324, "longitude": 87.8615, "zone": "Lower Gangetic"},
    {"district": "Cuttack", "state": "Odisha", "latitude": 20.4625, "longitude": 85.8828, "zone": "East Coast Plains"},
    {"district": "Bhopal", "state": "Madhya Pradesh", "latitude": 23.2599, "longitude": 77.4126, "zone": "Central Plateau"},
    {"district": "Raipur", "state": "Chhattisgarh", "latitude": 21.2514, "longitude": 81.6296, "zone": "Eastern Plateau"},
    {"district": "Jodhpur", "state": "Rajasthan", "latitude": 26.2389, "longitude": 73.0243, "zone": "Western Dry"},
    {"district": "Anand", "state": "Gujarat", "latitude": 22.5645, "longitude": 72.9289, "zone": "Gujarat Plains"},
    {"district": "Akola", "state": "Maharashtra", "latitude": 20.7002, "longitude": 77.0082, "zone": "Vidarbha"},
    {"district": "Pune", "state": "Maharashtra", "latitude": 18.5204, "longitude": 73.8567, "zone": "Western Plateau"},
    {"district": "Guntur", "state": "Andhra Pradesh", "latitude": 16.3067, "longitude": 80.4365, "zone": "East Coast Plains"},
    {"district": "Warangal", "state": "Telangana", "latitude": 17.9689, "longitude": 79.5941, "zone": "Southern Plateau"},
    {"district": "Dharwad", "state": "Karnataka", "latitude": 15.4589, "longitude": 75.0078, "zone": "Northern Transition"},
    {"district": "Coimbatore", "state": "Tamil Nadu", "latitude": 11.0168, "longitude": 76.9558, "zone": "Southern Plateau"},
    {"district": "Palakkad", "state": "Kerala", "latitude": 10.7867, "longitude": 76.6548, "zone": "West Coast Wet"},
    {"district": "Kamrup", "state": "Assam", "latitude": 26.2006, "longitude": 91.5606, "zone": "Brahmaputra Valley"},
    {"district": "Shillong", "state": "Meghalaya", "latitude": 25.5788, "longitude": 91.8933, "zone": "Eastern Himalayan"},
    {"district": "Dimapur", "state": "Nagaland", "latitude": 25.9068, "longitude": 93.7274, "zone": "Eastern Himalayan"},
    {"district": "Imphal West", "state": "Manipur", "latitude": 24.8170, "longitude": 93.9368, "zone": "Eastern Himalayan"},
    {"district": "Agartala", "state": "Tripura", "latitude": 23.8315, "longitude": 91.2868, "zone": "Eastern Himalayan"},
    {"district": "Gangtok", "state": "Sikkim", "latitude": 27.3389, "longitude": 88.6065, "zone": "Eastern Himalayan"}
]

NASA_POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMETERS = "T2M_MAX,T2M_MIN,T2M,RH2M,PRECTOTCORR,WS10M,WS10M_MAX,PS,ALLSKY_SFC_SW_DWN,T2MDEW"
START_DATE = "20210101"
END_DATE = "20241231"

def fetch_point(loc):
    params = {
        "parameters": PARAMETERS,
        "community": "AG",
        "longitude": loc["longitude"],
        "latitude": loc["latitude"],
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }
    s = requests.Session()
    for attempt in range(4):
        try:
            r = s.get(NASA_POWER_URL, params=params, timeout=35)
            if r.status_code == 200:
                p_data = r.json()["properties"]["parameter"]
                sub = pd.DataFrame(p_data).reset_index().rename(columns={"index": "date_str"})
                sub["date"] = pd.to_datetime(sub["date_str"], format="%Y%m%d")
                sub.drop(columns=["date_str"], inplace=True)
                sub["district"] = loc["district"]
                sub["state"] = loc["state"]
                sub["zone"] = loc["zone"]
                sub["latitude"] = loc["latitude"]
                sub["longitude"] = loc["longitude"]
                return sub
            time.sleep(1.0)
        except Exception:
            time.sleep(1.5)
    return None

print(f"1/3. Extracting 28 Pan-India states via NASA POWER (6 workers)...")
batch_dfs = []
with ThreadPoolExecutor(max_workers=6) as ex:
    futures = {ex.submit(fetch_point, loc): loc for loc in INDIAN_GRID_28}
    for f in tqdm(as_completed(futures), total=len(futures), desc="Fetching Weather"):
        res = f.result()
        if res is not None:
            batch_dfs.append(res)

if not batch_dfs:
    raise RuntimeError("Extraction failed. Please verify your internet connection.")

df_raw = pd.concat(batch_dfs, ignore_index=True)

# Standardize names & units
rename_dict = {
    "T2M": "temperature_2m_mean",
    "T2M_MIN": "temperature_2m_min",
    "T2M_MAX": "temperature_2m_max",
    "RH2M": "relative_humidity_2m_mean",
    "PRECTOTCORR": "precipitation_sum",
    "WS10M": "wind_speed_10m_mean",
    "WS10M_MAX": "wind_speed_10m_max",
    "PS": "surface_pressure_mean",
    "ALLSKY_SFC_SW_DWN": "shortwave_radiation_sum",
    "T2MDEW": "dew_point_2m_mean"
}
df_raw.rename(columns=rename_dict, inplace=True)
df_raw.replace(-999.0, np.nan, inplace=True)

df_raw["wind_speed_10m_max"] = df_raw["wind_speed_10m_max"] * 3.6
df_raw["wind_speed_10m_mean"] = df_raw["wind_speed_10m_mean"] * 3.6
df_raw["shortwave_radiation_sum"] = df_raw["shortwave_radiation_sum"].clip(lower=0.0)
df_raw["precipitation_sum"] = df_raw["precipitation_sum"].clip(lower=0.0)
df_raw["relative_humidity_2m_mean"] = df_raw["relative_humidity_2m_mean"].clip(lower=1.0, upper=100.0)

lead_cols = ["date", "district", "state", "zone", "latitude", "longitude"]
tail_cols = [c for c in df_raw.columns if c not in lead_cols]
df_clean = df_raw[lead_cols + tail_cols].sort_values(by=["district", "date"]).reset_index(drop=True)

# In-memory interpolation
numeric_cols = [c for c in tail_cols if c in df_clean.columns]
df_clean[numeric_cols] = df_clean.groupby("district")[numeric_cols].transform(
    lambda grp: grp.interpolate(method="linear").bfill().ffill()
)

print(f"Extracted {len(df_clean):,} daily observations across {df_clean['district'].nunique()} districts.")

# ------------------------------------------------------------------------------
# 2/3. AGROMET FEATURE ENGINEERING (IN-MEMORY)
# ------------------------------------------------------------------------------
print("2/3. Engineering rolling stress horizons & physical indices...")
df_clean["year"] = df_clean["date"].dt.year
df_clean["month"] = df_clean["date"].dt.month
df_clean["day_of_year"] = df_clean["date"].dt.dayofyear
df_clean["doy_sin"] = np.sin(2 * np.pi * df_clean["day_of_year"] / 365.25)
df_clean["doy_cos"] = np.cos(2 * np.pi * df_clean["day_of_year"] / 365.25)

def assign_season(m):
    return "KHARIF" if m in [6, 7, 8, 9, 10] else ("RABI" if m in [11, 12, 1, 2, 3] else "ZAID")

df_clean["agro_season"] = df_clean["month"].apply(assign_season)

# VPD (kPa)
es = 0.61078 * np.exp((17.27 * df_clean["temperature_2m_mean"]) / (df_clean["temperature_2m_mean"] + 237.3))
ea = es * (df_clean["relative_humidity_2m_mean"] / 100.0)
df_clean["vapor_pressure_deficit"] = np.clip(es - ea, 0, None)

# Hargreaves-Samani ET0
t_diff = np.clip(df_clean["temperature_2m_max"] - df_clean["temperature_2m_min"], 0.1, None)
df_clean["et0_fao_evapotranspiration"] = (
    0.0023 * (df_clean["temperature_2m_mean"] + 17.8) * np.sqrt(t_diff) * (df_clean["shortwave_radiation_sum"] * 0.408)
).clip(lower=0.5, upper=12.0)

# Grouped Rolling Aggregations
grp = df_clean.groupby("district")
df_clean["rainfall_1d"] = df_clean["precipitation_sum"]
df_clean["rainfall_3d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_clean["rainfall_7d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(7, min_periods=1).sum())
df_clean["rainfall_14d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(14, min_periods=1).sum())

df_clean["temperature_3d_mean"] = grp["temperature_2m_mean"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["temperature_7d_mean"] = grp["temperature_2m_mean"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_clean["temperature_3d_max"] = grp["temperature_2m_max"].transform(lambda x: x.rolling(3, min_periods=1).max())
df_clean["temperature_7d_max"] = grp["temperature_2m_max"].transform(lambda x: x.rolling(7, min_periods=1).max())
df_clean["temperature_3d_min"] = grp["temperature_2m_min"].transform(lambda x: x.rolling(3, min_periods=1).min())

df_clean["humidity_3d_mean"] = grp["relative_humidity_2m_mean"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["humidity_7d_mean"] = grp["relative_humidity_2m_mean"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_clean["et0_3d_sum"] = grp["et0_fao_evapotranspiration"].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_clean["et0_7d_sum"] = grp["et0_fao_evapotranspiration"].transform(lambda x: x.rolling(7, min_periods=1).sum())
df_clean["wind_speed_3d_mean"] = grp["wind_speed_10m_max"].transform(lambda x: x.rolling(3, min_periods=1).mean())

def calc_streaks(series, condition_fn):
    flags = condition_fn(series).astype(int)
    streaks, cur = [], 0
    for v in flags:
        cur = cur + 1 if v == 1 else 0
        streaks.append(cur)
    return pd.Series(streaks, index=series.index)

df_clean["consecutive_dry_days"] = grp["precipitation_sum"].transform(lambda s: calc_streaks(s, lambda x: x < 1.0))
df_clean["consecutive_rain_days"] = grp["precipitation_sum"].transform(lambda s: calc_streaks(s, lambda x: x >= 2.5))

# ------------------------------------------------------------------------------
# 3/3. ICAR AGRONOMIC ALIGNMENT & TARGET FORMULATION
# ------------------------------------------------------------------------------
print("3/3. Formulating ICAR stage-specific target labels...")
ICAR_CROP_MATRIX = {
    "Wheat": {
        "season": "RABI",
        "stages": ["Vegetative", "Flowering", "Grain_Filling", "Maturity"],
        "temp_opt_min": 12.0, "temp_opt_max": 25.0, "heat_limit": 32.0,
        "water_need_7d": 25.0
    },
    "Rice": {
        "season": "KHARIF",
        "stages": ["Transplanting", "Tillering", "Flowering", "Maturity"],
        "temp_opt_min": 20.0, "temp_opt_max": 33.0, "heat_limit": 38.0,
        "water_need_7d": 45.0
    },
    "Cotton": {
        "season": "KHARIF",
        "stages": ["Vegetative", "Square_Formation", "Boll_Formation", "Maturity"],
        "temp_opt_min": 21.0, "temp_opt_max": 32.0, "heat_limit": 39.0,
        "water_need_7d": 28.0
    },
    "Maize": {
        "season": "KHARIF",
        "stages": ["Vegetative", "Tasseling", "Silking", "Maturity"],
        "temp_opt_min": 18.0, "temp_opt_max": 30.0, "heat_limit": 36.0,
        "water_need_7d": 30.0
    },
    "Mustard": {
        "season": "RABI",
        "stages": ["Vegetative", "Flowering", "Pod_Formation", "Maturity"],
        "temp_opt_min": 10.0, "temp_opt_max": 22.0, "heat_limit": 30.0,
        "water_need_7d": 18.0
    },
    "Soybean": {
        "season": "KHARIF",
        "stages": ["Vegetative", "Pod_Initiation", "Pod_Filling", "Maturity"],
        "temp_opt_min": 20.0, "temp_opt_max": 30.0, "heat_limit": 37.0,
        "water_need_7d": 28.0
    }
}

aligned = []
for crop_name, p in ICAR_CROP_MATRIX.items():
    sub = df_clean[df_clean["agro_season"] == p["season"]].copy()
    sub["crop"] = crop_name

    if p["season"] == "RABI":
        s_map = {11: p["stages"][0], 12: p["stages"][1], 1: p["stages"][1], 2: p["stages"][2], 3: p["stages"][3]}
    else:
        s_map = {6: p["stages"][0], 7: p["stages"][0], 8: p["stages"][1], 9: p["stages"][2], 10: p["stages"][3]}

    sub["crop_stage"] = sub["month"].map(s_map).fillna(p["stages"][0])
    sub["temp_opt_min"] = p["temp_opt_min"]
    sub["temp_opt_max"] = p["temp_opt_max"]
    sub["heat_limit"] = p["heat_limit"]
    sub["water_need_7d"] = p["water_need_7d"]
    aligned.append(sub)

df_ml = pd.concat(aligned, ignore_index=True)

# TARGET 1: Heat Stress
c_heat = [
    (df_ml["temperature_3d_max"] >= df_ml["heat_limit"] + 2.0),
    (df_ml["temperature_3d_max"] >= df_ml["heat_limit"])
]
df_ml["heat_stress"] = np.select(c_heat, ["HIGH", "MODERATE"], default="LOW")

# TARGET 2: Waterlogging Risk
c_waterlog = [
    (df_ml["rainfall_3d_sum"] > 110.0) & (df_ml["crop"] != "Rice"),
    (df_ml["rainfall_3d_sum"] > 65.0) & (df_ml["crop"] != "Rice")
]
df_ml["waterlogging_risk"] = np.select(c_waterlog, ["HIGH", "MODERATE"], default="LOW")

# TARGET 3: Irrigation Requirement
water_deficit = df_ml["et0_7d_sum"] - df_ml["rainfall_7d_sum"]
c_irrig = [
    (water_deficit > 35.0) & (df_ml["rainfall_3d_sum"] < 2.0),
    (water_deficit > 20.0) & (df_ml["rainfall_3d_sum"] < 8.0),
    (water_deficit > 10.0) & (df_ml["rainfall_3d_sum"] < 15.0)
]
df_ml["irrigation"] = np.select(
    c_irrig,
    ["URGENT_IRRIGATION", "IRRIGATION_REQUIRED", "LIGHT_IRRIGATION"],
    default="NO_IRRIGATION"
)
df_ml.loc[df_ml["rainfall_3d_sum"] >= 20.0, "irrigation"] = "NO_IRRIGATION"

# TARGET 4: Spraying Suitability (CIBRC / ASABE Standards)
cond_spray_unsafe = (
    (df_ml["wind_speed_10m_max"] > 16.0) |
    (df_ml["precipitation_sum"] > 1.0) |
    (df_ml["temperature_2m_max"] > 35.0)
)
cond_spray_wait = (
    (df_ml["wind_speed_10m_max"] < 3.0) |
    (df_ml["relative_humidity_2m_mean"] > 85.0) |
    (df_ml["relative_humidity_2m_mean"] < 35.0)
)
df_ml["spraying_suitability"] = np.where(
    cond_spray_unsafe, "UNSAFE",
    np.where(cond_spray_wait, "WAIT", "SUITABLE")
)

# TARGET 5: Composite Crop Weather Risk
pts = (
    (df_ml["heat_stress"] == "HIGH").astype(int) * 3 +
    (df_ml["heat_stress"] == "MODERATE").astype(int) * 1 +
    (df_ml["waterlogging_risk"] == "HIGH").astype(int) * 3 +
    (df_ml["waterlogging_risk"] == "MODERATE").astype(int) * 1 +
    (df_ml["irrigation"] == "URGENT_IRRIGATION").astype(int) * 2 +
    (df_ml["consecutive_dry_days"] > 21).astype(int) * 2
)
df_ml["crop_weather_risk"] = np.select(
    [pts >= 5, pts >= 3, pts >= 1],
    ["SEVERE", "HIGH", "MODERATE"],
    default="LOW"
)

# Persist both to disk AND keep active in-memory variable `df_ml`
df_ml.to_parquet("pan_india_ml_ready.parquet", engine="pyarrow", index=False)

print("\n" + "=" * 60)
print(f"PIPELINE COMPLETED SUCCESSFULLY!")
print(f"Total ML Training Samples: {len(df_ml):,} rows x {df_ml.shape[1]} columns")
print("In-memory variable `df_ml` is ready for model training.")
print("=" * 60)

1/3. Extracting 28 Pan-India states via NASA POWER (6 workers)...


Fetching Weather:   0%|          | 0/28 [00:00<?, ?it/s]

Extracted 40,908 daily observations across 28 districts.
2/3. Engineering rolling stress horizons & physical indices...
3/3. Formulating ICAR stage-specific target labels...

PIPELINE COMPLETED SUCCESSFULLY!
Total ML Training Samples: 102,424 rows x 51 columns
In-memory variable `df_ml` is ready for model training.


In [ ]:
# ==============================================================================
# CELL 1: STEPS 9-13 — TEMPORAL SPLITTING & 4-MODEL BENCHMARKING
# ==============================================================================
!pip install -q xgboost lightgbm catboost scikit-learn

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# 1. Select Feature Columns
feature_cols = [
    "latitude", "longitude",
    "temperature_2m_mean", "temperature_2m_min", "temperature_2m_max",
    "relative_humidity_2m_mean", "precipitation_sum", "wind_speed_10m_max",
    "surface_pressure_mean", "shortwave_radiation_sum", "dew_point_2m_mean",
    "vapor_pressure_deficit", "et0_fao_evapotranspiration",
    "rainfall_3d_sum", "rainfall_7d_sum", "rainfall_14d_sum",
    "temperature_3d_mean", "temperature_7d_mean", "temperature_3d_max",
    "humidity_3d_mean", "humidity_7d_mean", "et0_7d_sum", "wind_speed_3d_mean",
    "consecutive_dry_days", "consecutive_rain_days",
    "doy_sin", "doy_cos", "month"
]

# 2. Categorical Encoding
cat_encoders = {}
for col in ["crop", "crop_stage", "agro_season"]:
    le = LabelEncoder()
    df_ml[col + "_enc"] = le.fit_transform(df_ml[col].astype(str))
    cat_encoders[col] = le
    feature_cols.append(col + "_enc")

# 3. Temporal Train/Test Split (Avoids Data Leakage)
train_mask = df_ml["year"] <= 2023
test_mask = df_ml["year"] == 2024

X_train = df_ml.loc[train_mask, feature_cols]
X_test = df_ml.loc[test_mask, feature_cols]

print(f"Features: {len(feature_cols)} | Training Rows: {len(X_train):,} | Test Rows: {len(X_test):,}")

# 4. Benchmark on Target: crop_weather_risk
benchmark_target = "crop_weather_risk"
target_le = LabelEncoder()
y_train = target_le.fit_transform(df_ml.loc[train_mask, benchmark_target])
y_test = target_le.transform(df_ml.loc[test_mask, benchmark_target])

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_seed=42, verbose=0)
}

benchmark_rows = []

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    if preds.ndim > 1:
        preds = preds.ravel()

    acc = accuracy_score(y_test, preds)
    p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)

    benchmark_rows.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(p, 4),
        "Recall": round(r, 4),
        "F1-Score": round(f1, 4)
    })

comparison_df = pd.DataFrame(benchmark_rows)
print("\n" + "=" * 60)
print(f"BENCHMARK RESULTS: {benchmark_target.upper()}")
print("=" * 60)
print(comparison_df.to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
Features: 31 | Training Rows: 76,776 | Test Rows: 25,648

BENCHMARK RESULTS: CROP_WEATHER_RISK
        Model  Accuracy  Precision  Recall  F1-Score
Random Forest    0.9826     0.9824  0.9664    0.9741
      XGBoost    0.9925     0.9848  0.9760    0.9802
     LightGBM    0.9939     0.9857  0.9787    0.9820
     CatBoost    0.9918     0.9883  0.9765    0.9822


In [ ]:
# ==============================================================================
# CELL 2: STEPS 14-18 — MULTI-TARGET PRODUCTION RUN, SAVE & PREDICTOR
# ==============================================================================
import joblib
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

ALL_TARGETS = [
    "crop_weather_risk",
    "irrigation",
    "heat_stress",
    "waterlogging_risk",
    "spraying_suitability"
]

target_encoders = {}
production_models = {}

print("Training dedicated production models across all 5 targets...")

for tgt in ALL_TARGETS:
    le = LabelEncoder()
    y_tr = le.fit_transform(df_ml.loc[train_mask, tgt])
    y_te = le.transform(df_ml.loc[test_mask, tgt])
    target_encoders[tgt] = le

    clf = lgb.LGBMClassifier(
        n_estimators=120,
        max_depth=7,
        learning_rate=0.08,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    clf.fit(X_train, y_tr)
    preds = clf.predict(X_test)
    production_models[tgt] = clf

    print(f"\n--- Evaluation for Target: {tgt} ---")
    print(classification_report(y_te, preds, target_names=le.classes_, zero_division=0))

# 1. Serialize all components to a single portable file
artifacts = {
    "models": production_models,
    "target_encoders": target_encoders,
    "cat_encoders": cat_encoders,
    "feature_cols": feature_cols
}

joblib.dump(artifacts, "farmer_advisory_pipeline.pkl")
print("\n" + "=" * 60)
print("Pipeline serialized successfully to: farmer_advisory_pipeline.pkl")
print("=" * 60)

# 2. Standalone Inference Function
def predict_farmer_advisory(farmer_input: dict) -> dict:
    """
    Stand-alone inference function for farmer crop-weather advisory.
    Takes raw operational and weather inputs; outputs multi-target risk labels
    and high-risk probability alerts without any backend/FastAPI dependencies.
    """
    pipeline = joblib.load("farmer_advisory_pipeline.pkl")
    models = pipeline["models"]
    t_encs = pipeline["target_encoders"]
    c_encs = pipeline["cat_encoders"]
    f_cols = pipeline["feature_cols"]

    inp = farmer_input.copy()

    # Cyclical day-of-year features
    month = inp.get("month", 7)
    doy = month * 30
    inp["month"] = month
    inp["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    inp["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    # Indian Agro-Season determination
    if "agro_season" not in inp:
        inp["agro_season"] = "KHARIF" if month in [6, 7, 8, 9, 10] else ("RABI" if month in [11, 12, 1, 2, 3] else "ZAID")

    # Vapor Pressure Deficit (VPD in kPa)
    if "vapor_pressure_deficit" not in inp:
        t_mean = inp.get("temperature_2m_mean", (inp["temperature_2m_max"] + inp["temperature_2m_min"]) / 2)
        inp["temperature_2m_mean"] = t_mean
        es_val = 0.61078 * np.exp((17.27 * t_mean) / (t_mean + 237.3))
        ea_val = es_val * (inp["relative_humidity_2m_mean"] / 100.0)
        inp["vapor_pressure_deficit"] = max(0.0, es_val - ea_val)

    # Encode categorical entries
    for col in ["crop", "crop_stage", "agro_season"]:
        enc = c_encs[col]
        val = str(inp[col])
        inp[col + "_enc"] = enc.transform([val])[0] if val in enc.classes_ else 0

    # Build evaluation DataFrame
    row = pd.DataFrame([inp])
    for col in f_cols:
        if col not in row.columns:
            row[col] = 0.0
    row = row[f_cols]

    advisories = {}
    probabilities = {}

    for tgt, model in models.items():
        enc = t_encs[tgt]
        pred_idx = model.predict(row)[0]
        advisories[tgt] = enc.inverse_transform([pred_idx])[0]

        # Extract probabilities for key alerts
        probs = model.predict_proba(row)[0]
        for c_name, p_val in zip(enc.classes_, probs):
            if c_name in ["HIGH", "SEVERE", "URGENT_IRRIGATION", "UNSAFE"]:
                probabilities[f"{tgt}_{c_name.lower()}_probability"] = round(float(p_val), 3)

    return {
        "advisories": advisories,
        "risk_probabilities": probabilities
    }

Training dedicated production models across all 5 targets...

--- Evaluation for Target: crop_weather_risk ---
              precision    recall  f1-score   support

        HIGH       0.97      0.98      0.98      2564
         LOW       1.00      1.00      1.00     17644
    MODERATE       0.99      0.99      0.99      4380
      SEVERE       0.99      0.95      0.97      1060

    accuracy                           0.99     25648
   macro avg       0.99      0.98      0.98     25648
weighted avg       0.99      0.99      0.99     25648


--- Evaluation for Target: irrigation ---
                     precision    recall  f1-score   support

IRRIGATION_REQUIRED       0.97      0.98      0.98      1984
   LIGHT_IRRIGATION       0.99      0.99      0.99      5536
      NO_IRRIGATION       1.00      1.00      1.00     17968
  URGENT_IRRIGATION       0.85      0.84      0.85       160

           accuracy                           0.99     25648
          macro avg       0.95      0.95   

In [ ]:
# ==============================================================================
# STEP 19: STANDALONE INFERENCE VALIDATION ON REAL INDIAN AGRO SCENARIOS
# ==============================================================================

# Scenario A: Flowering Wheat in Bihar facing a hot, dry spell
scenario_bihar_wheat = {
    "latitude": 25.59,
    "longitude": 85.13,
    "crop": "Wheat",
    "crop_stage": "Flowering",
    "month": 2,
    "temperature_2m_max": 34.6,
    "temperature_2m_min": 19.8,
    "temperature_2m_mean": 27.2,
    "relative_humidity_2m_mean": 58.0,
    "precipitation_sum": 0.0,
    "wind_speed_10m_max": 19.2,
    "surface_pressure_mean": 99.4,
    "shortwave_radiation_sum": 21.8,
    "dew_point_2m_mean": 18.2,
    "et0_fao_evapotranspiration": 5.4,
    "rainfall_3d_sum": 0.0,
    "rainfall_7d_sum": 0.0,
    "rainfall_14d_sum": 2.5,
    "temperature_3d_mean": 27.8,
    "temperature_7d_mean": 26.5,
    "temperature_3d_max": 34.6,
    "humidity_3d_mean": 60.0,
    "humidity_7d_mean": 58.0,
    "et0_7d_sum": 37.8,
    "wind_speed_3d_mean": 17.5,
    "consecutive_dry_days": 19,
    "consecutive_rain_days": 0
}

# Scenario B: Tillering Rice in Coastal Odisha under heavy monsoon downpour
scenario_odisha_rice = {
    "latitude": 20.46,
    "longitude": 85.88,
    "crop": "Rice",
    "crop_stage": "Tillering",
    "month": 8,
    "temperature_2m_max": 29.0,
    "temperature_2m_min": 24.5,
    "temperature_2m_mean": 26.8,
    "relative_humidity_2m_mean": 92.0,
    "precipitation_sum": 45.0,
    "wind_speed_10m_max": 24.0,
    "surface_pressure_mean": 98.2,
    "shortwave_radiation_sum": 12.0,
    "dew_point_2m_mean": 25.0,
    "et0_fao_evapotranspiration": 2.8,
    "rainfall_3d_sum": 115.0,
    "rainfall_7d_sum": 180.0,
    "rainfall_14d_sum": 240.0,
    "temperature_3d_mean": 27.0,
    "temperature_7d_mean": 27.2,
    "temperature_3d_max": 30.0,
    "humidity_3d_mean": 90.0,
    "humidity_7d_mean": 88.0,
    "et0_7d_sum": 19.5,
    "wind_speed_3d_mean": 21.0,
    "consecutive_dry_days": 0,
    "consecutive_rain_days": 6
}

for title, scenario in [("Scenario A: Bihar Wheat (Heat & Drought Stress)", scenario_bihar_wheat),
                       ("Scenario B: Odisha Rice (Active Monsoon)", scenario_odisha_rice)]:
    output = predict_farmer_advisory(scenario)
    print("=" * 60)
    print(title)
    print("=" * 60)
    for category, val in output["advisories"].items():
        print(f"• {category:<25}: {val}")
    print("\nCritical Risk Probabilities:")
    for alert, prob in output["risk_probabilities"].items():
        print(f"• {alert:<35}: {prob:.1%}")
    print()

Scenario A: Bihar Wheat (Heat & Drought Stress)
• crop_weather_risk        : HIGH
• irrigation               : IRRIGATION_REQUIRED
• heat_stress              : HIGH
• waterlogging_risk        : LOW
• spraying_suitability     : UNSAFE

Critical Risk Probabilities:
• crop_weather_risk_high_probability : 99.6%
• crop_weather_risk_severe_probability: 0.3%
• irrigation_urgent_irrigation_probability: 0.0%
• heat_stress_high_probability       : 100.0%
• waterlogging_risk_high_probability : 0.0%
• spraying_suitability_unsafe_probability: 100.0%

Scenario B: Odisha Rice (Active Monsoon)
• crop_weather_risk        : LOW
• irrigation               : NO_IRRIGATION
• heat_stress              : LOW
• waterlogging_risk        : LOW
• spraying_suitability     : UNSAFE

Critical Risk Probabilities:
• crop_weather_risk_high_probability : 0.0%
• crop_weather_risk_severe_probability: 0.0%
• irrigation_urgent_irrigation_probability: 0.0%
• heat_stress_high_probability       : 0.0%
• waterlogging_risk_high_

In [ ]:
!pip install -q geopy requests

In [ ]:
# ==============================================================================
# CELL 1: ROBUST RE-TRAIN WITH EXACT FEATURE CONTRACTS
# ==============================================================================
import joblib
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# 1. Exact list of 28 numeric features + 3 encoded categorical features
CORE_NUMERIC_FEATURES = [
    "latitude", "longitude",
    "temperature_2m_mean", "temperature_2m_min", "temperature_2m_max",
    "relative_humidity_2m_mean", "precipitation_sum", "wind_speed_10m_max",
    "surface_pressure_mean", "shortwave_radiation_sum", "dew_point_2m_mean",
    "vapor_pressure_deficit", "et0_fao_evapotranspiration",
    "rainfall_1d", "rainfall_3d_sum", "rainfall_7d_sum", "rainfall_14d_sum",
    "temperature_3d_mean", "temperature_7d_mean", "temperature_3d_max",
    "temperature_7d_max", "temperature_3d_min",
    "humidity_3d_mean", "humidity_7d_mean", "et0_3d_sum", "et0_7d_sum",
    "wind_speed_3d_mean", "consecutive_dry_days", "consecutive_rain_days",
    "doy_sin", "doy_cos", "month"
]

CATEGORICAL_COLS = ["crop", "crop_stage", "agro_season"]

# Encode categoricals in df_ml
cat_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_ml[col + "_enc"] = le.fit_transform(df_ml[col].astype(str))
    cat_encoders[col] = le

ALL_MODEL_FEATURES = CORE_NUMERIC_FEATURES + [c + "_enc" for c in CATEGORICAL_COLS]

# Train/Test split
train_mask = df_ml["year"] <= 2023
test_mask = df_ml["year"] == 2024

X_train = df_ml.loc[train_mask, ALL_MODEL_FEATURES].astype(np.float32)
X_test = df_ml.loc[test_mask, ALL_MODEL_FEATURES].astype(np.float32)

TARGETS = [
    "crop_weather_risk",
    "irrigation",
    "heat_stress",
    "waterlogging_risk",
    "spraying_suitability"
]

trained_models = {}
target_encoders = {}

print("Training production models with strict feature locking...")
for tgt in TARGETS:
    t_le = LabelEncoder()
    y_tr = t_le.fit_transform(df_ml.loc[train_mask, tgt])
    target_encoders[tgt] = t_le

    clf = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    clf.fit(X_train, y_tr)
    trained_models[tgt] = clf
    print(f"✓ Model trained for {tgt} (Classes: {list(t_le.classes_)})")

# Save the exact metadata and ordering
pipeline_bundle = {
    "models": trained_models,
    "target_encoders": target_encoders,
    "cat_encoders": cat_encoders,
    "numeric_features": CORE_NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_COLS,
    "all_features": ALL_MODEL_FEATURES
}

joblib.dump(pipeline_bundle, "farmer_advisory_pipeline.pkl")
print("\n" + "=" * 60)
print("SUCCESS: Pipeline trained & saved to farmer_advisory_pipeline.pkl")
print("=" * 60)

Training production models with strict feature locking...
✓ Model trained for crop_weather_risk (Classes: ['HIGH', 'LOW', 'MODERATE', 'SEVERE'])
✓ Model trained for irrigation (Classes: ['IRRIGATION_REQUIRED', 'LIGHT_IRRIGATION', 'NO_IRRIGATION', 'URGENT_IRRIGATION'])
✓ Model trained for heat_stress (Classes: ['HIGH', 'LOW', 'MODERATE'])
✓ Model trained for waterlogging_risk (Classes: ['HIGH', 'LOW', 'MODERATE'])
✓ Model trained for spraying_suitability (Classes: ['SUITABLE', 'UNSAFE', 'WAIT'])

SUCCESS: Pipeline trained & saved to farmer_advisory_pipeline.pkl


In [ ]:
# ==============================================================================
# CELL 2: VERIFIED PLACE-TO-ADVISORY INFERENCE ENGINE
# ==============================================================================
import joblib
import requests
import numpy as np
import pandas as pd
from datetime import datetime
from geopy.geocoders import Nominatim

def get_live_weather_metrics(lat: float, lon: float) -> dict:
    """Queries live + 14-day past weather and computes agrometeorological horizons."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": [
            "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
            "precipitation_sum", "relative_humidity_2m_mean", "wind_speed_10m_max",
            "shortwave_radiation_sum", "et0_fao_evapotranspiration"
        ],
        "past_days": 14,
        "forecast_days": 1,
        "timezone": "Asia/Kolkata"
    }

    resp = requests.get(url, params=params, timeout=25)
    resp.raise_for_status()
    d = resp.json()["daily"]
    df = pd.DataFrame(d).fillna(method="ffill").fillna(method="bfill")

    # Current day
    curr = df.iloc[-1]

    # Aggregations
    r1 = float(curr["precipitation_sum"] or 0.0)
    r3 = float(df["precipitation_sum"].iloc[-3:].sum())
    r7 = float(df["precipitation_sum"].iloc[-7:].sum())
    r14 = float(df["precipitation_sum"].iloc[-14:].sum())

    t_mean = float(curr["temperature_2m_mean"])
    t_max = float(curr["temperature_2m_max"])
    t_min = float(curr["temperature_2m_min"])
    rh = float(curr["relative_humidity_2m_mean"])

    t3_mean = float(df["temperature_2m_mean"].iloc[-3:].mean())
    t7_mean = float(df["temperature_2m_mean"].iloc[-7:].mean())
    t3_max = float(df["temperature_2m_max"].iloc[-3:].max())
    t7_max = float(df["temperature_2m_max"].iloc[-7:].max())
    t3_min = float(df["temperature_2m_min"].iloc[-3:].min())

    h3_mean = float(df["relative_humidity_2m_mean"].iloc[-3:].mean())
    h7_mean = float(df["relative_humidity_2m_mean"].iloc[-7:].mean())

    et0_val = float(curr["et0_fao_evapotranspiration"] or 4.2)
    et0_3d = float(df["et0_fao_evapotranspiration"].iloc[-3:].sum())
    et0_7d = float(df["et0_fao_evapotranspiration"].iloc[-7:].sum())

    wind_max = float(curr["wind_speed_10m_max"] or 8.5)
    wind3_mean = float(df["wind_speed_10m_max"].iloc[-3:].mean())

    # Streaks
    p_hist = df["precipitation_sum"].tolist()
    dry_streak, rain_streak = 0, 0
    for p in reversed(p_hist):
        if p < 1.0: dry_streak += 1
        else: break
    for p in reversed(p_hist):
        if p >= 2.5: rain_streak += 1
        else: break

    # VPD
    es = 0.61078 * np.exp((17.27 * t_mean) / (t_mean + 237.3))
    ea = es * (rh / 100.0)
    vpd = max(0.0, es - ea)

    return {
        "temperature_2m_mean": t_mean,
        "temperature_2m_min": t_min,
        "temperature_2m_max": t_max,
        "relative_humidity_2m_mean": rh,
        "precipitation_sum": r1,
        "wind_speed_10m_max": wind_max,
        "surface_pressure_mean": 101.3,
        "shortwave_radiation_sum": float(curr["shortwave_radiation_sum"] or 19.5),
        "dew_point_2m_mean": t_mean - ((100.0 - rh) / 5.0),
        "vapor_pressure_deficit": vpd,
        "et0_fao_evapotranspiration": et0_val,
        "rainfall_1d": r1,
        "rainfall_3d_sum": r3,
        "rainfall_7d_sum": r7,
        "rainfall_14d_sum": r14,
        "temperature_3d_mean": t3_mean,
        "temperature_7d_mean": t7_mean,
        "temperature_3d_max": t3_max,
        "temperature_7d_max": t7_max,
        "temperature_3d_min": t3_min,
        "humidity_3d_mean": h3_mean,
        "humidity_7d_mean": h7_mean,
        "et0_3d_sum": et0_3d,
        "et0_7d_sum": et0_7d,
        "wind_speed_3d_mean": wind3_mean,
        "consecutive_dry_days": dry_streak,
        "consecutive_rain_days": rain_streak
    }

def get_farmer_advisory_for_place(place_name: str, crop: str = "Rice", crop_stage: str = "Flowering"):
    # 1. Geocode
    geo = Nominatim(user_agent="farmer_advisory_agent_india")
    loc = geo.geocode(f"{place_name}, India")
    if not loc:
        raise ValueError(f"Could not resolve '{place_name}'. Use a prominent district name.")

    lat, lon = loc.latitude, loc.longitude

    # 2. Weather
    w = get_live_weather_metrics(lat, lon)

    # 3. Load Model Artifacts
    bundle = joblib.load("farmer_advisory_pipeline.pkl")
    models = bundle["models"]
    t_encs = bundle["target_encoders"]
    c_encs = bundle["cat_encoders"]
    all_feats = bundle["all_features"]

    # 4. Assemble Exact Input Vector
    payload = w.copy()
    payload["latitude"] = lat
    payload["longitude"] = lon

    # Season & calendar alignment
    if crop in ["Wheat", "Mustard"]:
        agro_season = "RABI"
        month_val = 1
    else:
        agro_season = "KHARIF"
        month_val = 8

    payload["crop"] = crop
    payload["crop_stage"] = crop_stage
    payload["agro_season"] = agro_season
    payload["month"] = month_val
    doy = month_val * 30
    payload["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    payload["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    # Encode Categoricals
    for c in ["crop", "crop_stage", "agro_season"]:
        enc = c_encs[c]
        val = str(payload[c])
        payload[c + "_enc"] = enc.transform([val])[0] if val in enc.classes_ else 0

    # Build strict single-row DataFrame ordered by all_feats
    input_row = pd.DataFrame([payload])[all_feats].astype(np.float32)

    # 5. Predict
    advisories = {}
    probabilities = {}
    for tgt, model in models.items():
        t_enc = t_encs[tgt]
        pred_idx = model.predict(input_row)[0]
        advisories[tgt] = t_enc.inverse_transform([pred_idx])[0]

        probs = model.predict_proba(input_row)[0]
        for class_name, prob in zip(t_enc.classes_, probs):
            probabilities[f"{tgt}_{class_name.lower()}"] = round(float(prob), 4)

    # 6. Actionable Output
    print("=" * 70)
    print("🌾 FARMER CROP-WEATHER ADVISORY REPORT")
    print("=" * 70)
    print(f"📍 Location      : {loc.address}")
    print(f"🌱 Crop & Stage  : {crop} ({crop_stage}) | Season: {agro_season}")
    print(f"⛅ Live Weather  : Temp Max: {w['temperature_2m_max']:.1f}°C | Rain 3d: {w['rainfall_3d_sum']:.1f} mm | Wind: {w['wind_speed_10m_max']:.1f} km/h")
    print(f"💧 Moisture State: 7-Day ET₀: {w['et0_7d_sum']:.1f} mm | 7-Day Rain: {w['rainfall_7d_sum']:.1f} mm | Dry Days: {w['consecutive_dry_days']}")
    print("-" * 70)
    print("🤖 MODEL PREDICTIONS:")
    for k, v in advisories.items():
        print(f"  • {k:<25}: {v}")

    print("\n🚨 MODEL PREDICTION PROBABILITIES:")
    for k, v in probabilities.items():
        if any(term in k for term in ["high", "urgent", "unsafe", "moderate", "required"]):
            print(f"  • {k:<36}: {v:.1%}")

    print("-" * 70)
    print("📋 ACTIONABLE FARMER ADVISORY:")

    # Irrigation
    irr = advisories["irrigation"]
    if irr == "URGENT_IRRIGATION":
        print("  🚨 **Urgent Irrigation**: Moisture deficit is critical. Irrigate field immediately to prevent yield loss.")
    elif irr == "IRRIGATION_REQUIRED":
        print(f"  💧 **Irrigation Advised**: High crop water demand. Plan standard irrigation within 24–48 hours.")
    elif irr == "LIGHT_IRRIGATION":
        print("  🚿 **Light Watering**: Surface topsoil is dry; apply light sprinkler or drip top-up.")
    else:
        print("  ✅ **No Irrigation**: Soil moisture reserves are currently sufficient.")

    # Spraying
    spray = advisories["spraying_suitability"]
    if spray == "UNSAFE":
        print(f"  ⛔ **Spraying Unsafe**: Unfavorable wind ({w['wind_speed_10m_max']:.1f} km/h) or precipitation risk. Do not apply chemicals.")
    elif spray == "WAIT":
        print(f"  ⏳ **Wait to Spray**: High humidity or stagnant air. Wait for clearer afternoon conditions.")
    else:
        print("  🎯 **Optimal Spray Window**: Wind velocity and humidity are safe for pesticide/fertilizer spray.")

    # Heat Stress
    heat = advisories["heat_stress"]
    if heat == "HIGH":
        print(f"  🔥 **Heat Stress Warning**: Daytime heat ({w['temperature_2m_max']:.1f}°C) exceeds safe thresholds for {crop}.")
    elif heat == "MODERATE":
        print(f"  🌡️ **Moderate Heat**: Temperatures are elevated. Ensure adequate soil hydration.")

    # Waterlogging
    wl = advisories["waterlogging_risk"]
    if wl in ["HIGH", "MODERATE"]:
        print(f"  🌊 **Drainage Warning**: Rain accumulation ({w['rainfall_3d_sum']:.1f} mm). Ensure all field outlets are clear.")

    print("=" * 70 + "\n")

In [ ]:
# 1. Jodhpur, Rajasthan (Arid western belt - high irrigation & heat stress test)
get_farmer_advisory_for_place(
    place_name="Jodhpur, Rajasthan",
    crop="Cotton",
    crop_stage="Boll_Formation"
)

# 2. Palakkad, Kerala (High humidity/coastal-monsoon test)
get_farmer_advisory_for_place(
    place_name="Palakkad, Kerala",
    crop="Rice",
    crop_stage="Tillering"
)

# 3. Ludhiana, Punjab (Northern wheat belt test)
get_farmer_advisory_for_place(
    place_name="Ludhiana, Punjab",
    crop="Wheat",
    crop_stage="Flowering"
)

/tmp/ipykernel_3229/881722859.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = pd.DataFrame(d).fillna(method="ffill").fillna(method="bfill")


🌾 FARMER CROP-WEATHER ADVISORY REPORT
📍 Location      : Jodhpur, Jodhpur Tehsil, Jodhpur, Rajasthan, India
🌱 Crop & Stage  : Cotton (Boll_Formation) | Season: KHARIF
⛅ Live Weather  : Temp Max: 35.6°C | Rain 3d: 3.1 mm | Wind: 13.5 km/h
💧 Moisture State: 7-Day ET₀: 42.3 mm | 7-Day Rain: 3.1 mm | Dry Days: 0
----------------------------------------------------------------------
🤖 MODEL PREDICTIONS:
  • crop_weather_risk        : LOW
  • irrigation               : LIGHT_IRRIGATION
  • heat_stress              : LOW
  • waterlogging_risk        : LOW
  • spraying_suitability     : UNSAFE

🚨 MODEL PREDICTION PROBABILITIES:
  • crop_weather_risk_high              : 0.0%
  • crop_weather_risk_moderate          : 0.3%
  • irrigation_irrigation_required      : 0.0%
  • irrigation_urgent_irrigation        : 0.0%
  • heat_stress_high                    : 0.0%
  • heat_stress_moderate                : 1.1%
  • waterlogging_risk_high              : 0.0%
  • waterlogging_risk_moderate          : 0.

/tmp/ipykernel_3229/881722859.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = pd.DataFrame(d).fillna(method="ffill").fillna(method="bfill")


🌾 FARMER CROP-WEATHER ADVISORY REPORT
📍 Location      : Palakkad, Kerala, India
🌱 Crop & Stage  : Rice (Tillering) | Season: KHARIF
⛅ Live Weather  : Temp Max: 32.6°C | Rain 3d: 3.5 mm | Wind: 19.0 km/h
💧 Moisture State: 7-Day ET₀: 36.4 mm | 7-Day Rain: 9.5 mm | Dry Days: 1
----------------------------------------------------------------------
🤖 MODEL PREDICTIONS:
  • crop_weather_risk        : LOW
  • irrigation               : NO_IRRIGATION
  • heat_stress              : LOW
  • waterlogging_risk        : LOW
  • spraying_suitability     : UNSAFE

🚨 MODEL PREDICTION PROBABILITIES:
  • crop_weather_risk_high              : 0.0%
  • crop_weather_risk_moderate          : 0.0%
  • irrigation_irrigation_required      : 0.0%
  • irrigation_urgent_irrigation        : 0.0%
  • heat_stress_high                    : 0.0%
  • heat_stress_moderate                : 0.0%
  • waterlogging_risk_high              : 0.0%
  • waterlogging_risk_moderate          : 0.0%
  • spraying_suitability_unsafe   

/tmp/ipykernel_3229/881722859.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = pd.DataFrame(d).fillna(method="ffill").fillna(method="bfill")


In [ ]:
get_farmer_advisory_for_place(
    place_name="buxar",
    crop="wheat",
    crop_stage="Tasseling"
)

🌾 FARMER CROP-WEATHER ADVISORY REPORT
📍 Location      : Buxar, Bihar, India
🌱 Crop & Stage  : wheat (Tasseling) | Season: KHARIF
⛅ Live Weather  : Temp Max: 31.9°C | Rain 3d: 14.0 mm | Wind: 19.3 km/h
💧 Moisture State: 7-Day ET₀: 27.0 mm | 7-Day Rain: 30.7 mm | Dry Days: 0
----------------------------------------------------------------------
🤖 MODEL PREDICTIONS:
  • crop_weather_risk        : LOW
  • irrigation               : NO_IRRIGATION
  • heat_stress              : LOW
  • waterlogging_risk        : LOW
  • spraying_suitability     : UNSAFE

🚨 MODEL PREDICTION PROBABILITIES:
  • crop_weather_risk_high              : 0.0%
  • crop_weather_risk_moderate          : 0.0%
  • irrigation_irrigation_required      : 0.0%
  • irrigation_urgent_irrigation        : 0.0%
  • heat_stress_high                    : 0.0%
  • heat_stress_moderate                : 0.0%
  • waterlogging_risk_high              : 0.0%
  • waterlogging_risk_moderate          : 0.0%
  • spraying_suitability_unsafe    

/tmp/ipykernel_3229/881722859.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = pd.DataFrame(d).fillna(method="ffill").fillna(method="bfill")


In [ ]:
get_farmer_advisory_for_place(
    place_name="guntur",
    crop="rice",
    crop_stage="Flowering"
)

🌾 FARMER CROP-WEATHER ADVISORY REPORT
📍 Location      : Guntur, Andhra Pradesh, 522001, India
🌱 Crop & Stage  : rice (Flowering) | Season: KHARIF
⛅ Live Weather  : Temp Max: 30.4°C | Rain 3d: 13.7 mm | Wind: 10.8 km/h
💧 Moisture State: 7-Day ET₀: 32.2 mm | 7-Day Rain: 21.2 mm | Dry Days: 0
----------------------------------------------------------------------
🤖 MODEL PREDICTIONS:
  • crop_weather_risk        : LOW
  • irrigation               : NO_IRRIGATION
  • heat_stress              : LOW
  • waterlogging_risk        : LOW
  • spraying_suitability     : UNSAFE

🚨 MODEL PREDICTION PROBABILITIES:
  • crop_weather_risk_high              : 0.0%
  • crop_weather_risk_moderate          : 0.0%
  • irrigation_irrigation_required      : 0.0%
  • irrigation_urgent_irrigation        : 0.0%
  • heat_stress_high                    : 0.0%
  • heat_stress_moderate                : 0.0%
  • waterlogging_risk_high              : 0.0%
  • waterlogging_risk_moderate          : 0.0%
  • spraying_suita

/tmp/ipykernel_3229/881722859.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = pd.DataFrame(d).fillna(method="ffill").fillna(method="bfill")


In [ ]:
from google.colab import files

files.download('farmer_advisory_pipeline.pkl')